In [1]:
from transformers import AutoModel, AutoTokenizer
from peft import PeftModel, PeftConfig
import re
import json
from tqdm import tqdm
import pickle

/home/lc/.conda/envs/pytorch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
import json

In [2]:
dic = {
    "事件类型": "货币政策调整",
    "事件元素": {
        "主体": "中国人民银行",
        "时间": "2023年9月15日",
        "动作": "下调",
        "对象": "金融机构存款准备金率",
        "幅度": "0.25个百分点",
        "目标": "7.4%",
        "例外情况": "金融机构中已执行5%存款准备金率的除外"
    }
}

In [5]:
print(json.dumps(dic, ensure_ascii=False, indent=4))

{
    "事件类型": "货币政策调整",
    "事件元素": {
        "主体": "中国人民银行",
        "时间": "2023年9月15日",
        "动作": "下调",
        "对象": "金融机构存款准备金率",
        "幅度": "0.25个百分点",
        "目标": "7.4%",
        "例外情况": "金融机构中已执行5%存款准备金率的除外"
    }
}


In [2]:
class args:
    peft_model_path = '/home/lc/projects/LLaMA-Factory/outputs/oaast_sft_zh/checkpoint-500'

In [3]:
config = PeftConfig.from_pretrained(args.peft_model_path)

In [4]:
config

LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='/home/lc/projects/pretrained_models/chatglm3-6b', revision=None, task_type='CAUSAL_LM', inference_mode=True, r=8, target_modules={'query_key_value'}, lora_alpha=16, lora_dropout=0.1, fan_in_fan_out=False, bias='none', modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={})

In [5]:
base_model = AutoModel.from_pretrained(config.base_model_name_or_path, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path, trust_remote_code=True)

Loading checkpoint shards: 100%|██████████| 7/7 [00:16<00:00,  2.34s/it]


In [6]:
model = PeftModel.from_pretrained(base_model, args.peft_model_path)
model = model.merge_and_unload()

In [7]:
model = model.to('cuda:2')

In [10]:
query = '扫雷代码'
resp, his = model.chat(tokenizer=tokenizer, query=query)

In [13]:
inputs = tokenizer.build_chat_input(query, history=[], role='user')

In [17]:
DEFAULT_SYSTEM_PROMPT = '''
You are ChatGLM3, a large language model trained by Zhipu.AI. Follow the user's instructions carefully. Respond using markdown.
'''.strip()

In [18]:
chat_history = [{
    'role': 'system',
    'content': DEFAULT_SYSTEM_PROMPT,
}]

In [23]:
query = '扫雷代码'
resp, his = model.chat(tokenizer=tokenizer, query=query, history=chat_history)

In [24]:
resp

'扫雷代码是一个用于扫雷游戏（Minesweeper）的Python代码。这个游戏是一个经典的计算机游戏，玩家需要在一个有地雷的网格中找出所有的非地雷元素。\n\n下面是一个简单的Python实现，其中包含了扫雷游戏的基本逻辑：\n\n```python\nimport random\n\nclass Minesweeper:\n    def __init__(self, rows, columns):\n        self.rows = rows\n        self.columns = columns\n        self.grid = [[random.randint(0, 255) for _ in range(columns)] for _ in range(rows)]\n\n    def count_mines(self):\n        return sum(row.count(255) for row in self.grid)\n\n    def count_safe(self):\n        return sum(sum(row.count(255) for row in self.grid) for _ in range(self.rows))\n\n    def count_board(self):\n        return sum(sum(row.count(255) for row in self.grid) for _ in range(self.rows))\n\n    def count_mines_on_board(self):\n        return sum(row.count(255) for row in self.grid)\n\n    def mark_safe(self, row, column):\n        self.grid[row][column] = 0\n\n    def mark_mines(self, row, column):\n        self.grid[row][column] = 255\n\n    def find_mines(self, row, column):\n        if self.grid[row][

In [25]:
input_text = '税负转嫁：税收负担转嫁的简称,亦称税收转嫁,是指纳税人将所缴纳的税款,利用各种手段转移给他人负担。税负转嫁有广义和狭义之分,广义的税负转嫁包括税负转移和最终归宿两个部分,狭义的税负转嫁仅指纳税人把税收负担转移给他人的过程。税负转嫁的途径和形式主要有税负前转、税负后转、税负消转、税负散转、税收资本化等。税负转嫁的主要手段是变动价格,向前转嫁主要通过提高商品的销售价格,向后转嫁主要通过降低各生产要素价格,包括降低劳动者工资等。税法中规定的纳税人并不一定是最后的负税人,其间税收可能经过多次的转移。'

data_dict = {
    "instruction": "请抽取文本中的三元组并以结构化形式输出",
    "input": input_text
}
input_text = f"Instruction: {data_dict.get('instruction', '')}\nInput: {data_dict.get('input', '')}"

In [26]:
resp, his = model.chat(tokenizer=tokenizer, query=input_text, temperature=0.2, repetition_penalty=1.0)

In [27]:
print(resp)

[('税负转嫁', '别名', '税收负担转嫁'), ('税负转嫁', '分类', '广义税负转嫁'), ('税负转嫁', '分类', '狭义税负转嫁'), ('税负转嫁', '途径', '税负前转'), ('税负转嫁', '途径', '税负后转'), ('税负转嫁', '途径', '税负消转'), ('税负转嫁', '途径', '税负散转'), ('税负转嫁', '主要手段', '变动价格'), ('税负转嫁', '主要手段', '变动商品销售价格'), ('税负转嫁', '主要手段', '降低生产要素价格'), ('税负转嫁', '主要手段', '降低劳动者工资')]


In [ ]:
# for response, history, past_key_values in model.stream_chat(tokenizer=tokenizer, query=input_text, temperature=0.2, return_past_key_values=True):
#     print(response)

In [2]:
with open('new_eval_result.pickle', 'rb') as f:
    eval_result = pickle.load(f)

In [5]:
with open('new_eval_result.json', 'w') as f:
    f.write(json.dumps(eval_result, ensure_ascii=False, indent=4))

In [35]:
for single in eval_result:
    model_output = single['model_output']

    try:
        relation_list = list(set(eval(model_output)))

    except Exception:
        print(single)
    #     spos = re.findall(r'(\(.*?\))', model_output)
    #     relation_list = []

    #     for spo in spos:
    #         t = spo.split(',')
    #         if len(t) == 3:
    #             s, p, o = t
    #             s = s.replace(' ', '').replace("'", "").replace('(', '')
    #             p = p.replace(' ', '').replace("'", "")
    #             o = o.replace(' ', '').replace("'", "").replace(')', '')

    #             relation_list.append((s, p, o))

    #     relation_list = list(set(relation_list))

    # single['relation_list'] = relation_list

{'categories': ['税收理论', '税收基础'], 'word': '税负转嫁因素', 'text': '制约税收负担能否转嫁、转嫁方向和转嫁程度的因素,主要有三类:(1)税种。一般认为,直接课自商品的税,即流转税或间接税,容易转嫁;直接课于企业利润或个人收入的税,即收益税或直接税,不易转嫁。严格地讲,不同税种转嫁的可能性只有难易之分,不存在绝对不能转嫁的问题。实际上,即使是直接税,在一定条件下也可以实现一部分转嫁。如企业缴纳的所得税,在可能降低工资、延长工时、提高劳动强度的条件下,就有转嫁的可能。(2)课税商品供给与需求的相对弹性。课之于商品的税收一般不会出现完全转嫁或完全不能转嫁的问题,大量发生的是纳税人和其他各自负担一定的比例。纳税人自己负担部分和转嫁出去部分的比例受制于课税商品的相对弹性。需求弹性较大,供给弹性较小,税收将主要由纳税人承担;需求弹性较小,供给弹性较大,税收将主要由其他人负担。(3)课税商品生产与销售的竞争程度。一般认为,垄断性商品要比竞争性商品的税负转嫁容易一些,理论上不排除独占性商品的税负有完全转嫁出去的可能性。另外,纳税企业对某种生产资料在使用上的垄断程度,还决定着税负向后转嫁的可能性和转嫁程度;劳动者是否团结及其供应情况,决定着纳税人利用降低工资转嫁税款的可能性及程度;而课税范围、课税方法、生产周期、成本变动等等,也会影响税负转嫁。', 'model_output': "[('税负转嫁因素', '制约', '税收负担转嫁'), ('税负转嫁因素', '包括', '税种'.), ('税负转嫁因素', '包括', '课税商品供给与需求的相对弹性'), ('税负转嫁因素', '包括', '课税商品生产与销售的竞争程度'')]", 'relation_list': [('税负转嫁因素', '包括', '课税商品供给与需求的相对弹性'), ('税负转嫁因素', '包括', '税种.'), ('税负转嫁因素', '包括', '课税商品生产与销售的竞争程度'), ('税负转嫁因素', '制约', '税收负担转嫁')]}
{'categories': ['税收理论', '税收基础'], 'word': '税收负担能力', 'text': '纳税人负担税收的能力,是制定税率、确定纳税人税负的基本依据之一。关于衡量税收负担能力的标准,西方经济

In [36]:
with open('final_eval_result.json', 'w') as f:
    f.write(json.dumps(eval_result, ensure_ascii=False, indent=4))

In [34]:
eval_result

[{'categories': ['税收理论', '税收基础'],
  'word': '税收',
  'text': '历史上称“赋税”、“租税”、“捐税”,也简称“税”。英文 taxation,含有负担、重负之意。国家为了实现其职能,满足社会公共需要,按照法律规定的标准,强制地、无偿地取得财政收入的有效形式。税收收入是国家凭借政治权力通过参与国民收入分配和再分配而获得的。税收是一个历史范畴,是人类社会经济发展到一定阶段,有了剩余产品,产生了私有财产,出现了阶级、国家之后产生的。恩格斯说:“为了维护这种公共权力,就需要公民缴纳费用捐税。”税收不仅是政府组织财政收入的最佳、最有效的手段,而且在优化资源配置、贯彻产业政策、实现公平分配、促进经济增长等方面具有重要的调节作用,是政府实施宏观经济调控的工具和市场配置经济资源的必要补充。',
  'model_output': "[('税收', '别名', '赋税'), ('税收', '别名', '租税'), ('税收', '别名', '捐税'), ('税收', '含义', '负担'), ('税收', '含义', '重负'), ('税收', '英文', 'tuation'), ('税收', '含义', '促进经济增长'), ('税收', '作用', '优化资源配置'), ('税收', '作用', '贯彻产业政策'), ('税收', '作用', '促进公平分配'), ('税收', '作用', '促进社会公平'), ('税收', '作用', '促进社会和谐'), ('税收', '作用', '促进社会稳定')]",
  'relation_list': [('税收', '含义', '促进经济增长'),
   ('税收', '作用', '促进社会稳定'),
   ('税收', '作用', '贯彻产业政策'),
   ('税收', '别名', '赋税'),
   ('税收', '别名', '捐税'),
   ('税收', '含义', '负担'),
   ('税收', '别名', '租税'),
   ('税收', '作用', '促进公平分配'),
   ('税收', '作用', '促进社会公平'),
   ('税收', '作用', '促进社会和谐'),
   ('税收', '英文', 'tuation'),
   ('税收', '含义'

In [37]:
len(eval_result)

2999

In [28]:
model_output = "[('税收', '别名', '赋税'), ('税收', '别名', '租税'), ('税收', '别名', '捐税'), ('税收', '含义', '负担'), ('税收', '含义', '重负'), ('税收', '英文', 'tuation'), ('税收', '含义', '促进经济增长'), ('税收', '作用', '优化资源配置'), ('税收', '作用', '贯彻产业政策'), ('税收', '作用', '促进公平分配'), ('税收', '作用', '促进社会公平'), ('税收', '作用', '促进社会和谐'), ('税收', '作用', '促进社会稳定')]"

In [31]:
list(set(eval(model_output)))

[('税收', '含义', '促进经济增长'),
 ('税收', '作用', '促进社会稳定'),
 ('税收', '作用', '贯彻产业政策'),
 ('税收', '别名', '赋税'),
 ('税收', '别名', '捐税'),
 ('税收', '含义', '负担'),
 ('税收', '别名', '租税'),
 ('税收', '作用', '促进公平分配'),
 ('税收', '作用', '促进社会公平'),
 ('税收', '作用', '促进社会和谐'),
 ('税收', '英文', 'tuation'),
 ('税收', '含义', '重负'),
 ('税收', '作用', '优化资源配置')]

In [16]:
spos = re.findall(r'(\(.*?\))', model_output)

In [25]:
new_spos = []

for spo in spos:
    t = spo.split(',')
    if len(t) == 3:
        s, p, o = t
        s = s.replace(' ', '').replace("'", "").replace('(', '')
        p = p.replace(' ', '').replace("'", "")
        o = o.replace(' ', '').replace("'", "").replace(')', '')

        new_spos.append((s, p, o))       

In [26]:
new_spos

[('税收', '别名', '赋税'),
 ('税收', '别名', '捐税'),
 ('税收', '含义', '负担'),
 ('税收', '含义', '重负'),
 ('税收', '英文', 'tuation'),
 ('税收', '含义', '促进经济增长'),
 ('税收', '作用', '优化资源配置'),
 ('税收', '作用', '贯彻产业政策'),
 ('税收', '作用', '促进公平分配'),
 ('税收', '作用', '促进社会公平'),
 ('税收', '作用', '促进社会和谐'),
 ('税收', '作用', '促进社会稳定')]

In [19]:
t = spos[0].split(',')

In [20]:
s, p, o = t

In [22]:
s.replace(' ', '').replace("'", "").replace('(', '')

'税收'

In [21]:
s

"('税收'"

In [17]:
spos

["('税收', '别名', '赋税')",
 "('税收', '别名', '租税', '租税')",
 "('税收, '别名', '捐税')",
 "('税收', '含义', '负担')",
 "('税收', '含义', '重负')",
 "('税收', '英文', 'tuation')",
 "('税收', '含义', '促进经济增长')",
 "('税收', '作用', '优化资源配置')",
 "('税收', '作用', '贯彻产业政策')",
 "('税收', '作用', '促进公平分配')",
 "('税收', '作用', '促进社会公平')",
 "('税收', '作用', '促进社会和谐')",
 "('税收', '作用', '促进社会稳定')"]

In [14]:
eval(model_output)

SyntaxError: unterminated string literal (detected at line 1) (<string>, line 1)

In [29]:
re.findall(r'(\(.*?\))', resp)

['(税负转嫁, 包括, 税负转移)',
 '(税负转嫁, 包括, 最终归宿)',
 '(税负转嫁, 包括, 税负消转)',
 '(税负转嫁, 包括, 税收资本化)',
 '(税负转嫁, 主要手段, 变动价格)',
 '(税负转嫁, 主要手段, 提高商品销售价格)',
 '(税负转嫁, 主要手段, 降低各生产要素价格)',
 '(税负转嫁, 主要手段, 降低劳动者工资)']

In [2]:
with open('eval_data.json', 'r') as f:
    eval_data = json.load(f)

In [8]:
for single in tqdm(eval_data[:50]):
    word, text = single['word'], single['text']

    input_text = f'{word}：{text}'

    data_dict = {
        "instruction": "请抽取文本中的三元组并以结构化形式输出",
        "input": input_text
    }
    input_text = f"Instruction: {data_dict.get('instruction', '')}\nInput: {data_dict.get('input', '')}"

    resp, his = model.chat(tokenizer=tokenizer, query=input_text, temperature=0.2, max_length=1024)

    single['model_output'] = resp

    # r_list = re.findall(r'(\(.*?\))', resp)
    # relation_list = []
    # for i in r_list:
    #     spo = i.split(', ')
    #     if len(spo) == 3:
    #         s, p, o = i.split(', ')
    #         relation_list.append([s.replace('(', ''), p, o.replace(')', '')])

    # single['relation_list'] = relation_list
    

100%|██████████| 50/50 [01:21<00:00,  1.62s/it]


In [16]:
for single in eval_data:
    try:
        model_output = eval(single['model_output'])
        for i in model_output:
            assert len(i) == 3
    except Exception as e:
        print(single['model_output'])

[('税制结构', '分类', '单一税制结构'), ('税制结构', '分类', '复合税制结构'), ('税制结构', '分类', '流转税'), ('税制结构', '分类', '以税种在税制中的地位不同', '所得税为主体的税制模式'), ('税制结构', '分类', '以税种在税制中的地位不同', '资源税为主体的税制模式'), ('税制结构', '分类', '以税收管理的权限不同', '中央税制结构'), ('税制结构', '分类', '以税收管理的权限不同', '地方税制结构')]
[('自动稳定器', '别名', '内在稳定器'), ('自动稳定器', '作用', '减轻收入和价格波动'), ('自动稳定器', '来源', '财政税收制度'), ('自动稳定器', '作用', '经济稳定'), ('自动稳定器', '作用', '缓和经济波动'), ('自动稳定器', '作用', '需求管理'), ('自动稳定器', '作用', '自动调节经济'), ('自动稳定器', '作用', '减轻所得税税额', '个人'), ('自动稳定器', '作用', '减轻所得税税额', '企业'), ('自动稳定器', '作用', '增加所得税税额', '个人'), ('自动稳定器', '作用', '增加所得税税额', '企业')]
[('中华法系', '别名', '中国法系'), ('中华法系', '别名', '中国法系'), ('中华法系', '特点'), ('法律以君主意志为主'), ('礼教', '最高原则'), ('刑法', '发达'), ('民法', '薄弱'), ('行政司法合一')]


In [8]:
with open('eval_result.json', 'w') as f:
    f.write(json.dumps(eval_data, ensure_ascii=False, indent=4))

In [48]:
with open('/home/lc/projects/LLM-Efficient-Tuning/data/tax_dict_data3.json', 'r') as f:
    temp = json.load(f)

len(temp)

406

136

In [38]:
single = eval_data[5]

word, text = single['word'], single['text']

input_text = f'{word}：{text}'

data_dict = {
    "instruction": "请抽取文本中的三元组并以结构化形式输出",
    "input": input_text
}
input_text = f"Instruction: {data_dict.get('instruction', '')}\nInput: {data_dict.get('input', '')}"

1. 训练数据：136条人工标注的数据，用微调好的模型去抽取270条数据
2. 人工校准这270条数据，然后总共得到了406条有标注的数据
3. 用406条数据再微调模型，用微调好的模型去抽取剩下的2999个词

In [11]:
with open('./inference_result.json', 'r') as f:
    inference_result = json.load(f)

In [19]:
from collections import OrderedDict

In [20]:
predicate = OrderedDict()

In [33]:
for i in inference_result:
    relation_list = i['relation_list']

    tag = False
    for r in relation_list:
        p = r[1]
        # predicate[p] = predicate.get(p, 0) + 1
        if '主管' == p:
            tag = True
    
    if tag:
        print(i)

{'categories': ['税收管理', '税务行政管理', '税务监察'], 'word': '干部监督联席会议制度', 'text': '由国家税务总局人事司、巡视办、监察局为主,根据议定事项的性质分别请办公厅、法规司、财务司、稽查局、机关纪委等部门参加的,针对税务系统领导班子和领导干部贯彻执行党的路线方针政策和国家税收法律法规、遵守廉洁自律规定等情况进行分析和监督的协调议事制度。', 'model_output': "[('干部监督联席会议制度', '主管', '国家税务总局人事司'), ('干部监督联席会议制度', '主管', '巡视办'), ('干部监督联席会议制度', '主管', '监察局'), ('干部监督联席会议制度', '参加部门', '办公厅'), ('干部监督联席会议制度', '参加部门', '法规司'), ('干部监督联席会议制度', '参加部门', '财务司'), ('干部监督联席会议制度', '参加部门', '稽查局'), ('干部监督联席会议制度', '参加部门', '机关纪委')]", 'relation_list': [['干部监督联席会议制度', '主管', '国家税务总局人事司'], ['干部监督联席会议制度', '主管', '监察局'], ['干部监督联席会议制度', '主管', '巡视办'], ['干部监督联席会议制度', '参加部门', '办公厅'], ['干部监督联席会议制度', '参加部门', '财务司'], ['干部监督联席会议制度', '参加部门', '机关纪委'], ['干部监督联席会议制度', '参加部门', '法规司'], ['干部监督联席会议制度', '参加部门', '稽查局']]}
{'categories': ['税收管理', '机构人事', '总局内部机构及直属单位'], 'word': '办公厅', 'text': '国家税务总局主管日常公务、文秘和局机关行政管理事务的综合职能部门。主要职责是:负责机关文电、机要、会务、档案、信访、保密和保卫等工作;承担税收宣传、政务公开和新闻发布工作;管理机关财务和其他行政事务。', 'model_output': "[('办公厅', '隶属于', '国家税务总局'), ('办公厅', '主管', '日常公务'), ('办公厅', '主管', '文秘'), 

In [26]:
sorted(predicate.items(), key=lambda d: d[1], reverse=True)

[('包括', 1706),
 ('关联', 1053),
 ('分类', 467),
 ('目的', 467),
 ('属于', 459),
 ('别名', 390),
 ('特点', 368),
 ('主要税种', 261),
 ('征收对象', 136),
 ('作用', 103),
 ('同义词', 102),
 ('对称', 100),
 ('特征', 86),
 ('类型', 83),
 ('内容', 79),
 ('形式', 67),
 ('用途', 61),
 ('职责', 56),
 ('适用于', 52),
 ('方法', 48),
 ('优点', 45),
 ('简称', 44),
 ('主管', 44),
 ('规定', 37),
 ('功能', 37),
 ('方式', 34),
 ('计算方法', 33),
 ('征税范围', 32),
 ('法律特征', 32),
 ('使用场景', 28),
 ('主要内容', 27),
 ('对象', 26),
 ('缺点', 26),
 ('原则', 23),
 ('行为', 22),
 ('研究内容', 21),
 ('英文名', 21),
 ('表现', 21),
 ('反映', 21),
 ('审查内容', 20),
 ('表现形式', 19),
 ('要求', 19),
 ('结果', 19),
 ('来源', 19),
 ('性质', 19),
 ('发布机构', 19),
 ('分为', 18),
 ('情况', 18),
 ('分类方法', 17),
 ('特性', 17),
 ('征税对象', 17),
 ('税率', 17),
 ('税目', 17),
 ('基础', 17),
 ('要素', 16),
 ('主要特点', 16),
 ('关系', 16),
 ('纳税人', 16),
 ('任务', 16),
 ('调整对象', 15),
 ('适用范围', 15),
 ('原因', 15),
 ('开征税种', 15),
 ('主要功能', 14),
 ('含义', 13),
 ('条件', 13),
 ('隶属于', 13),
 ('观点', 12),
 ('税种', 12),
 ('征收方式', 12),
 ('适用对象', 12),
 ('手段', 12),
 ('工作

In [23]:
sorted(predicate)

['',
 ' 解决问题',
 '三种表示方法',
 '下设',
 '不受企业经营数量变动影响',
 '不同',
 '不同之处',
 '不属于',
 '不得从销售额中减除',
 '不适用于',
 '与',
 '与企业固定资产',
 '与企业基本组织机构',
 '与其他审计的不同',
 '专业化水平',
 '业务',
 '业务主管部门',
 '业务范围',
 '个人所得税税前扣除标准',
 '中心内容',
 '中断',
 '主体',
 '主张',
 '主管',
 '主管部门',
 '主管预算单位',
 '主营业务',
 '主要',
 '主要业务',
 '主要任务',
 '主要内容',
 '主要功能',
 '主要实现',
 '主要审计原则',
 '主要形式',
 '主要手段',
 '主要执行者',
 '主要渠道',
 '主要特征',
 '主要特点',
 '主要用于',
 '主要用途',
 '主要研究对象',
 '主要税种',
 '主要纳税人',
 '主要组成部分',
 '主要职责',
 '主要适用于',
 '举办机构',
 '交易场所',
 '交易对象',
 '产业类型',
 '产品',
 '产生',
 '产生与发展',
 '产生原因',
 '产生时间',
 '付款方式',
 '付款期限',
 '代理',
 '代表',
 '代表学说',
 '以',
 '以前适用国家',
 '任务',
 '企业生产经营耗费',
 '企业类型',
 '优先于',
 '优先适用的效力',
 '优势',
 '优惠措施',
 '优点',
 '会刊',
 '会员',
 '会计准则',
 '会计对象',
 '会计核算',
 '会计科目',
 '会议主题',
 '传入中国',
 '传递方式',
 '估计方法',
 '体制',
 '体现',
 '体现了',
 '体现原则',
 '作用',
 '作者',
 '使用人群',
 '使用国家',
 '使用场景',
 '使用对象',
 '使用方式',
 '使用方法',
 '使用范围',
 '例子',
 '依含烃类比例',
 '依据',
 '侵犯',
 '侵犯客体',
 '侵犯的客体',
 '便于',
 '保证范围',
 '倡导者',
 '倾向',
 '假设',
 '做法',
 '停止',
 '停止使用',
 '停止使用时间',
 '停止征收时间',
 '停止时间'

In [12]:
inference_result

[{'categories': ['税收理论', '税收基础'],
  'word': '税收',
  'text': '历史上称“赋税”、“租税”、“捐税”,也简称“税”。英文 taxation,含有负担、重负之意。国家为了实现其职能,满足社会公共需要,按照法律规定的标准,强制地、无偿地取得财政收入的有效形式。税收收入是国家凭借政治权力通过参与国民收入分配和再分配而获得的。税收是一个历史范畴,是人类社会经济发展到一定阶段,有了剩余产品,产生了私有财产,出现了阶级、国家之后产生的。恩格斯说:“为了维护这种公共权力,就需要公民缴纳费用捐税。”税收不仅是政府组织财政收入的最佳、最有效的手段,而且在优化资源配置、贯彻产业政策、实现公平分配、促进经济增长等方面具有重要的调节作用,是政府实施宏观经济调控的工具和市场配置经济资源的必要补充。',
  'model_output': "[('税收', '别名', '赋税'), ('税收', '别名', '租税'), ('税收', '别名', '捐税'), ('税收', '含义', '负担'), ('税收', '含义', '重负'), ('税收', '英文', 'tuation'), ('税收', '含义', '促进经济增长'), ('税收', '作用', '优化资源配置'), ('税收', '作用', '贯彻产业政策'), ('税收', '作用', '促进公平分配'), ('税收', '作用', '促进社会公平'), ('税收', '作用', '促进社会和谐'), ('税收', '作用', '促进社会稳定')]",
  'relation_list': [['税收', '含义', '促进经济增长'],
   ['税收', '作用', '促进社会稳定'],
   ['税收', '作用', '贯彻产业政策'],
   ['税收', '别名', '赋税'],
   ['税收', '别名', '捐税'],
   ['税收', '含义', '负担'],
   ['税收', '别名', '租税'],
   ['税收', '作用', '促进公平分配'],
   ['税收', '作用', '促进社会公平'],
   ['税收', '作用', '促进社会和谐'],
   ['税收', '英文', 'tuation'],
   ['税收', '含义'

In [39]:
print(input_text)

Instruction: 请抽取文本中的三元组并以结构化形式输出
Input: 税法调整对象：税法所调整的社会关系种类。具体指税收活动中各方主体之间所发生的社会关系,包括税收分配关系和税收征纳关系。税法的调整对象是税法区别于其他法律的主要标准,正是税收关系性质、范围的特殊性,才使税法成为有别于其他法律的特定法律领域。


In [40]:
resp, his = model.chat(tokenizer=tokenizer, query=input_text, temperature=0.2, max_new_tokens=256, repetition_penalty=1.0)

Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [41]:
resp

"[('税法调整对象', '关联', '税法')]"

In [37]:
eval(resp)

[]

In [14]:
for response, history, past_key_values in model.stream_chat(tokenizer=tokenizer, query=input_text, temperature=0.2, repetition_penalty=1.1, return_past_key_values=True):
    print(response)

(
(税
(税,
(税,
(税, 包括
(税, 包括,
(税, 包括,
(税, 包括, 税收
(税, 包括, 税收),
(税, 包括, 税收), (
(税, 包括, 税收), (税
(税, 包括, 税收), (税,
(税, 包括, 税收), (税,
(税, 包括, 税收), (税, 包括
(税, 包括, 税收), (税, 包括,
(税, 包括, 税收), (税, 包括,
(税, 包括, 税收), (税, 包括, 赋
(税, 包括, 税收), (税, 包括, 赋),
(税, 包括, 税收), (税, 包括, 赋), (
(税, 包括, 税收), (税, 包括, 赋), (赋
(税, 包括, 税收), (税, 包括, 赋), (赋,
(税, 包括, 税收), (税, 包括, 赋), (赋,
(税, 包括, 税收), (税, 包括, 赋), (赋, 包括
(税, 包括, 税收), (税, 包括, 赋), (赋, 包括,
(税, 包括, 税收), (税, 包括, 赋), (赋, 包括,
(税, 包括, 税收), (税, 包括, 赋), (赋, 包括, 兵
(税, 包括, 税收), (税, 包括, 赋), (赋, 包括, 兵役
(税, 包括, 税收), (税, 包括, 赋), (赋, 包括, 兵役),
(税, 包括, 税收), (税, 包括, 赋), (赋, 包括, 兵役), (
(税, 包括, 税收), (税, 包括, 赋), (赋, 包括, 兵役), (税
(税, 包括, 税收), (税, 包括, 赋), (赋, 包括, 兵役), (税,
(税, 包括, 税收), (税, 包括, 赋), (赋, 包括, 兵役), (税,
(税, 包括, 税收), (税, 包括, 赋), (赋, 包括, 兵役), (税, 别
(税, 包括, 税收), (税, 包括, 赋), (赋, 包括, 兵役), (税, 别名
(税, 包括, 税收), (税, 包括, 赋), (赋, 包括, 兵役), (税, 别名,
(税, 包括, 税收), (税, 包括, 赋), (赋, 包括, 兵役), (税, 别名,
(税, 包括, 税收), (税, 包括, 赋), (赋, 包括, 兵役), (税, 别名, 课
(税, 包括, 税收), (税, 包括, 赋), (赋, 包括, 兵役), (税, 别名, 课征
(税

In [13]:
re.findall(r'(\(.*?\))', resp)

["('税', '意思', '税收')", "('税', '区别', '赋')"]